# A Current Capstone Project

Seattle Public Utility (SPU) needs to release a report on greenhouse gas emission regularly. 
Experts at SPU need to 
1. log the data (in different formats) received from other departments. 
2. process the data
3. visualize the processing results and generate a report

The current workflow is human labor intensive, time consuming, and error proning.

**Sponsor's request**: Make this efficient and render comparable performance to human experts.

**Question**: How can you help the sponsor to address the problem?

# Our focus: Automation & Scripting in Python


- Filesystem navigation with `pathlib`
- Pattern-based file searches with `glob` and `rglob`
- File manipulation with `shutil`
- Using command-line arguments `argparse`

## What is Automation?

Automation means writing Python code to handle repetitive tasks such as:

- Renaming or organizing files  
- Cleaning data  
- Batch converting file formats  
- Creating backups  
- Extracting and processing text  
- Merging datasets  

In [ ]:
# ice-berg index 

## Navigating the Filesystem with `pathlib`

`pathlib` provides:
- Path objects representing files/folders
- Easy functions for reading/writing files
- Tools for directory traversal


In [ ]:
from pathlib import Path

p = Path(".")
print("Current directory:", p.resolve())

print("Contents:")
for item in p.iterdir():
    print(" -", item, "(dir)" if item.is_dir() else "(file)")

In [ ]:
# read and write files

p = Path("example.txt")
if not p.exists():
    p.write_text("Hello world!\nThis is sample text.")

txt = p.read_text()
upper = txt.upper()

Path("output.txt").write_text(upper)
print("Done.")


## Pattern Matching with `glob`

We often need to find files matching patterns:
- `"*.txt"`
- `"*.csv"`

We use `.glob()` or `.rglob()` from `pathlib`.


In [ ]:
from pathlib import Path

for nb in Path(".").glob("*.ipynb"):
    print("Notebook:", nb)

# recursive search

for log in Path(".").rglob("*.log"):
    print(log)


### Example: Count Characters in `.txt` Files

Find all `.txt` files in the current directory and print `<filename>.txt: <xyz> characters`

In [ ]:
from pathlib import Path

for txt_file in Path(".").glob("*.txt"):
    char_count = len(txt_file.read_text())
    print(f"{txt_file.name}: {char_count} characters")


## File Management with `shutil`

`shutil` gives us high-level file operations:

- Copying files
- Removing directories
- Creating ZIP archives



### Copy files

In [ ]:

from pathlib import Path
import shutil

backup = Path("backup")
backup.mkdir(exist_ok=True)

for f in Path(".").glob("*.txt"):
    shutil.copy(f, backup / f.name)
    print("Copied:", f.name)


### Make a zip Archive

In [ ]:
import shutil

shutil.make_archive("archive/backup_archive", "zip", "backup")


### Remove a zip Archive

In [ ]:
shutil.rmtree("archive") # remove the archive folder

### Batch renaming

In [ ]:

from pathlib import Path

images = Path("images")
images.mkdir(exist_ok=True)

# Make demo files
for i in range(3):
    (images / f"image{i}.jpg").write_text("fake image data")


Renamed: photo_003.jpg as photo_001.jpg
Renamed: photo_002.jpg as photo_002.jpg
Renamed: photo_005.jpg as photo_003.jpg
Renamed: photo_004.jpg as photo_004.jpg
Renamed: photo_006.jpg as photo_005.jpg
Renamed: image1.jpg as photo_006.jpg
Renamed: image0.jpg as photo_007.jpg
Renamed: image2.jpg as photo_008.jpg


In [ ]:
for n, img in enumerate(images.glob("*.jpg"), start=1):
    new = images / f"photo_{n:03d}.jpg"
    img.rename(new)
    print("Renamed:", img.name, "as", new.name)


## What is a CLI Script?

A **CLI script** (Command Line Interface script) is a Python program that:

- runs from the terminal, like `python clean.py --ext txt --delete`
- uses **arguments** to control behavior

We will use `argparse` which automatically:
- parses arguments
- validates options
- creates help messages
- handles flags like `--delete`

See `src/hello.py` and `src/clean_file.py` for examples

## Collaborative Activity: Building a Mini Data Workflow for a Pizza Shop

Your pizza shop is growing fast! To make sense of your sales data, you’ll need help from three key roles:
**Data Engineer, Data Analyst, and Engineer.**

Form a group of **4 students**. Each person chooses **one role**. Work together to build a clean, insightful, and automated data pipeline.


**1. Data Engineer**

**Goal:** Prepare clean, analysis-ready data.

**Tasks:**

* Inspect the raw sales data and identify issues such as:

  * Inconsistent size labels (e.g., `"lg"`, `"Large"`, `"L"`)
  * Quantity or price stored as text instead of numbers
  * Inconsistent casing or spelling in categories
* Use Python to fix these problems.
* Save and document the cleaned dataset so the Data Analyst can understand and use it.
* Work with the Data Analyst to agree on the final clean dataset structure (column names, datatypes, formats).

**2. Data Analyst x2**

**Goals:** Turn the cleaned data into business insight. Communicate the data’s story through visuals.

**Tasks:**

* Review the cleaned dataset provided by the Data Engineer.
* Choose **one key business metric** that a pizza shop manager would care about.
* If the data needs to be reorganized to compute the metric, discuss requirements with the Data Engineer.
* Load the metric defined by the Data Analyst. Understand its meaning and how it should be interpreted.
* Create a simple, clean Python visualization. 
* Write a brief narrative explaining what the visualization reveals about the pizza shop’s performance.
* **Deliverable:** A clearly stated **north star metric** and a **short justification** for why it matters to the business.


**3. Engineer**

**Goal:** Bring everything together into a repeatable pipeline.

**Tasks:**

* Build an automated end-to-end workflow that:

  1. Loads the raw data
  2. Runs the Data Engineer’s cleaning steps
  3. Produces the Data Analyst’s metric
  4. Generates the Developer’s visualization
* Decide on a logical file and folder structure (e.g., `data/raw/`, `data/clean/`, `outputs/plots/`, etc.).
* Add exception handling to make the workflow robust (e.g., missing files, invalid values, incorrect column names).
* Ensure the final outputs are saved in the correct locations.

**4. Designer**
**Goal:** Shape the overall user experience and presentation quality.

**Tasks:**

* Sketch a dashboard layout that showcases:
  1. The north star metric
  2. The visualization(s)
  3. The story behind the data
  4. Provide UI/UX feedback on clarity, accessibility, and ease of understanding.

In [14]:
import pandas as pd
import os
from IPython.display import display  # Optional for table display

def load_csv_complete(file_path):
    # Read all rows safely
    df = pd.read_csv(file_path, skipinitialspace=True, engine='python', on_bad_lines='warn')
    
    # Strip spaces in string columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    
    # Convert numeric columns safely
    for col in ['quantity', 'unit_price', 'total']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
    
    # Normalize extra_cheese
    if 'extra_cheese' in df.columns:
        df['extra_cheese'] = df['extra_cheese'].astype(str).str.upper().replace(
            {'TRUE': True, 'FALSE': False, 'YES': True, 'NO': False, 'NAN': None}
        )
    
    # Fill missing totals: total = quantity × unit_price
    df['total'] = df.apply(
        lambda row: row['quantity'] * row['unit_price'] 
        if pd.isna(row['total']) or row['total'] == 0 else row['total'], axis=1
    )
    
    # Reset index to start from 1
    df.index = range(1, len(df) + 1)
    
    return df

def save_clean_data(file_path, folder_name="clean_data"):
    # Load and clean CSV
    df_clean = load_csv_complete(file_path)
    
    # Create folder if it doesn't exist
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    
    # Save cleaned CSV to folder
    clean_file_path = os.path.join(folder_name, os.path.basename(file_path))
    df_clean.to_csv(clean_file_path, index=False)
    
    print(f"Cleaned data saved to: {clean_file_path}")
    
    # Optional: display the table
    display(df_clean)
    
# Example usage
save_clean_data("sales_2025-11-24.csv")


Cleaned data saved to: clean_data\sales_2025-11-24.csv


,date,order_id,item_type,item_name,size,quantity,extra_cheese,unit_price,total
1,2025-11-24,1,pizza,bbq chicken,Medium,3.0,True,13.67,41.01
2,2025-11-24,1,salad,greek salad,nan,1.0,None,7.00,7.00
3,2025-11-24,2,pizza,Four cheese,lARGE,3.0,True,NaN,NaN
4,2025-11-24,2,drink,water,nan,1.0,None,1.50,1.50
5,2025-11-24,3,pizza,Veggie,lARGE,1.0,False,11.64,11.64
6,2025-11-24,3,salad,GARDEN SALAD,nan,1.0,None,5.50,5.50
7,2025-11-24,4,pizza,Veggie,L,3.0,True,9.79,29.37
8,2025-11-24,5,pizza,bbq chicken,L,4.0,True,13.57,54.28
9,2025-11-24,5,drink,water,nan,2.0,None,1.50,3.00
10,2025-11-24,6,pizza,bbq chicken,L,1.0,True,10.81,10.81
